# SAR description v2 — add-on for `sar_optic_description.ipynb`
Run in the **same Colab runtime** after your original notebook's `%%writefile common.py` and `%%writefile describe.py` cells (this add-on imports from them and changes neither). Needs the same GPU runtime and a trained `baseline` checkpoint on the Hub.

What changes vs v1: the VLM now sees the **SAR itself + the translation**, measured SAR facts, the dataset's terrain/season label and the ground scale; hedging becomes `[high]/[medium]/[low]` tags plus one Reliability line; sampled decoding; 800-token budget; truncation flag.

In [ ]:
# 1) token from Colab Secrets (key icon, left sidebar -> add HF_WRITE_TOKEN, enable notebook access). Do NOT paste tokens into cells.
import os, sys
%cd /content/sar_fyp
sys.path.insert(0, "/content/sar_fyp")
if not os.environ.get("HF_WRITE_TOKEN"):
    from google.colab import userdata
    os.environ["HF_WRITE_TOKEN"] = userdata.get("HF_WRITE_TOKEN")
assert os.environ["HF_WRITE_TOKEN"], "HF_WRITE_TOKEN missing"
EXP = "baseline"


In [ ]:
%%writefile describe_v2.py
"""
describe_v2.py -- drop-in upgrade for describe.py. Does NOT modify common.py / describe.py; imports
translate_sar_to_optical and _load_vlm from describe.py and replaces only the description step.

WHY v1 PRODUCED NEAR-IDENTICAL, WEAK DESCRIPTIONS (diagnosed from your saved outputs)
 1. The VLM only ever saw the blurry, dark translated image, so the only true thing it could say about it
    was "low-resolution, grainy, blurred". The real information (the SAR) was never shown to it.
 2. Dataset metadata you already have (terrain class, season) was never passed to the VLM.
 3. The prompt opened with a long caveat block and told the model to flag artifacts, so 45-75% of all
    sentences were boilerplate hedging.
 4. Greedy decoding on near-identical inputs gives near-identical text.
 5. max_new_tokens=450 truncated long outputs mid-sentence (e.g. the 'bridge'/'flood' sample).

WHAT v2 DOES
 * Shows the VLM BOTH images: the (despeckled, contrast-stretched) SAR and the translated optical.
 * Computes measured SAR facts (3x3 region brightness/texture, dark-smooth areas, bright returns, dominant
   linear direction) and injects them, so every description is anchored to this patch's pixels.
 * Tells the VLM the ground scale (256 px = 2.56 km at 10 m/px) so it knows what is resolvable.
 * Replaces per-sentence hedging with explicit [high]/[medium]/[low] confidence tags + one Reliability line.
 * Sampled decoding (seeded per id, so reproducible), larger token budget, truncation flag, one retry.
 * Optional optical_source="real": describe the REAL Sentinel-2 image the same way -> pseudo-reference
   captions for evaluation and for later LoRA fine-tuning (SARLANG-1M's 'modality transfer' recipe).

Everything here is a decision-support aid on partly synthetic imagery, not verified ground truth.
"""
import gc, json, re, time, zlib
from pathlib import Path

import numpy as np
from PIL import Image

from common import (CACHE, RUNS, load_index, load_split, load_run, build_text_embeds, log, seed_all,
                    preflight, sync_data_meta, require_hf_token, require_checkpoint, hf_pull_file,
                    hf_push_file, dev)
from describe import translate_sar_to_optical, _load_vlm

PATCH_PX, PATCH_KM = 256, 2.56            # SEN1-2 patches: 256x256 px at 10 m/px
GRID = [["top-left", "top-centre", "top-right"],
        ["middle-left", "centre", "middle-right"],
        ["bottom-left", "bottom-centre", "bottom-right"]]
TERRAIN_TEXT = {"agri": "agricultural land", "barrenland": "barren / sparsely vegetated land",
                "grassland": "grassland", "urban": "urban / built-up area"}
WARNING = ("Partly synthetic: Image 2 is a neural SAR-to-optical translation, not a photograph. "
           "Preliminary decision-support description, not verified ground truth.")

# ───────────────────────────── SAR preprocessing + measured facts ─────────────────────────────
def load_sar_gray(src):
    img = src if isinstance(src, Image.Image) else Image.open(src)
    return np.asarray(img.convert("L").resize((PATCH_PX, PATCH_PX), Image.BILINEAR), np.float32) / 255.


def lee_filter(x, size=5):
    """Classic Lee despeckle (approximation: data are already log-scaled 8-bit)."""
    from scipy.ndimage import uniform_filter
    m = uniform_filter(x, size); m2 = uniform_filter(x * x, size)
    v = np.maximum(m2 - m * m, 0)
    noise = float(np.median(v)) + 1e-8
    return m + (v / (v + noise)) * (x - m)


def sar_for_vlm(gray, out=512):
    """Despeckled, percentile-stretched, upsampled RGB-ified SAR for the vision tower (preprocessing
    measurably helps VLM captioning on SAR in the SARLANG-1M study)."""
    g = lee_filter(gray, 5)
    lo, hi = np.percentile(g, [2, 98])
    g = np.clip((g - lo) / max(hi - lo, 1e-6), 0, 1)
    return Image.fromarray((g * 255).astype(np.uint8)).convert("RGB").resize((out, out), Image.BICUBIC)


def _cell(y, x, H, W):
    return GRID[min(2, int(3 * y / H))][min(2, int(3 * x / W))]


def _span(mask):
    ys, xs = np.nonzero(mask); H, W = mask.shape
    a, b = _cell(ys.min(), xs.min(), H, W), _cell(ys.max(), xs.max(), H, W)
    return a if a == b else f"{a} to {b}"


def _largest(mask):
    from scipy.ndimage import label
    lab, n = label(mask)
    if n == 0:
        return None
    sizes = np.bincount(lab.ravel())[1:]
    return lab == (1 + int(np.argmax(sizes)))


def _direction_words(theta_deg):
    t = theta_deg % 180
    if t < 20 or t >= 160: return "roughly horizontal (left-right)"
    if 70 <= t < 110: return "roughly vertical (top-bottom)"
    if t < 70: return "running from top-left to bottom-right"
    return "running from top-right to bottom-left"


def sar_stats(gray):
    """Returns (facts_text, facts_dict). All levels are RELATIVE to this image (z-scores / ranks), which
    is what differentiates one patch from another; absolute DN is given once for context."""
    from scipy.ndimage import (uniform_filter, gaussian_filter, sobel, binary_opening, label,
                               center_of_mass)
    H, W = gray.shape
    g = lee_filter(gray, 5)
    gm, gs = float(g.mean()), float(g.std()) + 1e-6
    d = {"mean_dn": int(round(gm * 255)), "std_dn": int(round(gs * 255))}
    lines = [f"- Patch = {PATCH_KM} km x {PATCH_KM} km (10 m/px). Mean backscatter {d['mean_dn']}/255, "
             f"spread {d['std_dn']}/255 ({'highly' if gs / max(gm, 1e-6) > 0.35 else 'moderately' if gs / max(gm, 1e-6) > 0.18 else 'weakly'} heterogeneous scene)."]

    # 3x3 regions
    ys, xs = np.linspace(0, H, 4).astype(int), np.linspace(0, W, 4).astype(int)
    cells = []
    for r in range(3):
        for c in range(3):
            blk = g[ys[r]:ys[r + 1], xs[c]:xs[c + 1]]
            cells.append((GRID[r][c], (float(blk.mean()) - gm) / gs, float(blk.std())))
    sd_rank = np.argsort(np.argsort([c[2] for c in cells]))          # 0..8, high = rough/varied texture
    reg = []
    for (name, z, _), rk in zip(cells, sd_rank):
        lvl = ("much darker than the scene average" if z < -1.0 else "darker than average" if z < -0.4 else
               "about average brightness" if z < 0.4 else "brighter than average" if z < 1.0 else
               "much brighter than the scene average")
        tex = "varied/rough texture" if rk >= 6 else "smooth/homogeneous texture" if rk <= 2 else "moderate texture"
        reg.append(f"  {name}: {lvl} (z={z:+.1f}), {tex}")
    lines.append("- Regions (relative to this image):\n" + "\n".join(reg))
    d["regions"] = [(n, round(z, 2)) for n, z, _ in cells]

    # dark smooth areas (water / tarmac / bare flat ground / radar shadow candidates)
    loc_sd = np.sqrt(np.maximum(uniform_filter(g * g, 7) - uniform_filter(g, 7) ** 2, 0))
    dark = (g < gm - 0.9 * gs) & (loc_sd < np.median(loc_sd) * 1.2)
    dark = binary_opening(dark, structure=np.ones((3, 3)), iterations=2)
    big = _largest(dark)
    d["dark_frac"] = round(float(dark.mean()), 3)
    if big is not None and big.mean() > 0.004:
        yy, xx = np.nonzero(big); h, w = np.ptp(yy) + 1, np.ptp(xx) + 1
        shape = "elongated, river/road-like" if max(h, w) / max(1, min(h, w)) > 2.5 else "compact / blob-like"
        lines.append(f"- Dark smooth areas (water / tarmac / bare flat ground / radar-shadow candidates): "
                     f"{100 * dark.mean():.0f}% of frame; largest patch {100 * big.mean():.0f}% of frame, "
                     f"{shape}, spanning {_span(big)}.")
    else:
        lines.append("- Dark smooth areas: essentially none detected.")

    # extended bright/rough areas
    br = binary_opening(g > gm + 1.2 * gs, structure=np.ones((3, 3)), iterations=2)
    bb = _largest(br)
    d["bright_frac"] = round(float(br.mean()), 3)
    lines.append(f"- Extended bright/rough areas (built-up, dense/tall vegetation or steep relief): "
                 f"{100 * br.mean():.0f}% of frame" +
                 (f"; largest {100 * bb.mean():.0f}% spanning {_span(bb)}." if bb is not None and bb.mean() > 0.004 else "."))

    # compact very bright returns (candidate strong scatterers: metal, corner reflectors, building edges)
    s3 = uniform_filter(gray, 3)
    lab, n = label(s3 > s3.mean() + 3.5 * s3.std())
    if n:
        sizes = np.bincount(lab.ravel())[1:]
        keep = [i + 1 for i, s in enumerate(sizes) if 2 <= s <= 80]
    else:
        keep = []
    d["n_bright_returns"] = len(keep)
    if len(keep) >= 3:
        cnt = {}
        for i in keep:
            cy, cx = center_of_mass(lab == i); k = _cell(cy, cx, H, W); cnt[k] = cnt.get(k, 0) + 1
        top = sorted(cnt.items(), key=lambda kv: -kv[1])[:3]
        lines.append(f"- Compact very bright returns: {len(keep)} (measurement of bright pixel clusters, NOT object "
                     f"counts); most in " + ", ".join(f"{k} ({v})" for k, v in top) + ".")
    else:
        lines.append(f"- Compact very bright returns: {'a few' if keep else 'none'}; no strong point scatterers.")

    # dominant linear / directional texture (structure tensor on smoothed image)
    sg = gaussian_filter(g, 1.5)
    gx, gy = sobel(sg, axis=1), sobel(sg, axis=0)
    Jxx, Jyy, Jxy = float((gx * gx).sum()), float((gy * gy).sum()), float((gx * gy).sum())
    coh = float(np.sqrt((Jxx - Jyy) ** 2 + 4 * Jxy ** 2) / (Jxx + Jyy + 1e-9))
    edge_deg = (np.degrees(0.5 * np.arctan2(2 * Jxy, Jxx - Jyy)) + 90) % 180   # edge direction, y axis points down
    d["linear_coherence"], d["linear_deg"] = round(coh, 2), round(float(edge_deg), 1)
    if coh > 0.35:
        lines.append(f"- Dominant linear/directional structure: {_direction_words(edge_deg)}, "
                     f"coherence {coh:.2f} (strong: likely field rows, roads, ridges or a river).")
    elif coh > 0.18:
        lines.append(f"- Directional structure: weak, tendency {_direction_words(edge_deg)} (coherence {coh:.2f}).")
    else:
        lines.append(f"- Directional structure: none dominant (coherence {coh:.2f}); texture is isotropic/patchy.")
    return "\n".join(lines), d


# ───────────────────────────── prompts ─────────────────────────────
_SYSTEM_COMMON = f"""You are an experienced remote-sensing image analyst writing a caption for ONE specific {PATCH_KM} km x {PATCH_KM} km patch (256x256 px, 10 m per pixel).

You receive two images of the SAME patch plus measured statistics.
- Image 1: Sentinel-1 SAR backscatter, despeckled and contrast-stretched. This is a real observation. Dark = smooth surfaces (calm water, tarmac, bare flat ground) or radar shadow; mid-grey = fields/vegetation; bright = rough ground, dense or tall vegetation, buildings, metal and corner-reflecting structures. Remaining grain is noise, not ground texture.
{{image2}}
- 'Measured SAR facts' are computed from the pixels and are reliable. Use them to anchor WHERE things are.

Scale limits at 10 m: fields, roads, rivers, lakes, settlements and large industrial buildings are resolvable; individual houses, cars, people and small objects are not. Never claim them and never give counts of them.

Style rules:
1. Be specific to THIS patch. Every sentence must carry a concrete observation: where (use the 3x3 position words top-left ... bottom-right), what shape, roughly what fraction of the frame, how bright/rough.
2. No generic filler. Do not mention that the image is low-resolution, blurry, grainy, noisy or artificial anywhere except the Reliability section.
3. Instead of hedging every sentence, end each substantive claim with a confidence tag: [high], [medium] or [low]. Use [high] only when the SAR facts and both images agree. If unsure, give your best interpretation tagged [low]; do not refuse to describe.
4. Never state identities, intent, affiliations or exact measurements you cannot see.
5. Finish with a Reliability section (one or two sentences) naming what is least trustworthy in THIS patch (a specific region, colour, or an ambiguous feature).
This is a preliminary decision-support description of partly synthetic imagery, not verified ground truth."""

_IMAGE2 = {
    "translated": "- Image 2: an optical-looking image generated from Image 1 by a neural network. It is NOT a photograph: colours and brightness are largely guesses. Use it only as a hint for what materials/classes the shapes might be. Where it disagrees with the SAR, believe the SAR for geometry and texture.",
    "real": "- Image 2: the real Sentinel-2 true-colour image of the same patch (acquired on a different day, so crops, water level or illumination may differ from the SAR). Trust it for materials and colour; trust the SAR for what the radar saw.",
}

PURPOSES = {
    "general": dict(
        sections=["Scene", "Layout", "Linear features", "Water and built-up", "Reliability"],
        task=("Describe the patch in rich visual detail for a general reader: land-cover mix with approximate "
              "percentages, parcel/field pattern, vegetation density, relief clues, settlements, roads, water "
              "bodies and the apparent season (use the dataset label if given).")),
    "defense": dict(
        sections=["Scene", "Built infrastructure", "Transport and access", "Anomalies", "Reliability"],
        task=("Describe the patch for an imagery-analysis / situational-awareness reader: built-up areas and "
              "their layout, road/rail/bridge networks and access routes, large flat cleared or paved surfaces, "
              "compact very bright returns, and anything that contrasts with its surroundings. Describe only "
              "what is visible; do not infer identity, ownership or intent. Say 'not resolvable' for anything "
              "smaller than about 30 m.")),
    "flood": dict(
        sections=["Scene", "Water extent", "Affected land and roads", "Vegetation", "Reliability"],
        task=("Describe the patch for a flood / disaster-impact reader: extent and shape of dark smooth areas "
              "that may be water, how they relate to drainage, terrain and field edges, roads or settlements "
              "next to them, and vegetation state. IMPORTANT: this is a single date with no pre-flood "
              "reference, so permanent water, flooded fields and other smooth dark surfaces (tarmac, bare sand, "
              "radar shadow) cannot be separated with certainty. State which is most plausible here and why, "
              "with a confidence tag. This is not flood-extent mapping.")),
}


def build_prompt(purpose, terrain, season, facts, optical_source="translated"):
    assert purpose in PURPOSES, f"unknown purpose {purpose!r}; choices: {list(PURPOSES)}"
    p = PURPOSES[purpose]
    system = _SYSTEM_COMMON.format(image2=_IMAGE2[optical_source])
    meta = (f"Dataset label (coarse; may not describe the whole patch): terrain = "
            f"{TERRAIN_TEXT.get(terrain, 'unknown')}; season = {season or 'unknown'}.")
    user = (f"{meta}\n\nMeasured SAR facts:\n{facts}\n\nTask: {p['task']}\n\n"
            f"Write exactly these labelled sections, in order, each starting with its name and a colon: "
            f"{'; '.join(p['sections'])}. Total length about 250-350 words.")
    return system, user


def prompts_are_sane():
    for purpose in PURPOSES:
        for src in _IMAGE2:
            s, u = build_prompt(purpose, "agri", "fall", "- facts", src)
            assert "decision-support" in s and "[high]" in s and "Reliability" in s, purpose
            assert (src == "translated") == ("NOT a photograph" in s), (purpose, src)
    return True


# ───────────────────────────── generation ─────────────────────────────
_HEDGE = re.compile(r"cannot be confirmed|synthesis artifact|appears? to show|low[- ]resolution|blur|ambiguous|"
                    r"indistinct|unverif|cannot be determined|impossible to", re.I)


def _hedge_share(text):
    sents = [s for s in re.split(r"(?<=[.!?\]])\s+", text) if s.strip()]
    return sum(bool(_HEDGE.search(s)) for s in sents) / max(1, len(sents))


def _missing_sections(text, purpose):
    return [s for s in PURPOSES[purpose]["sections"] if not re.search(rf"(?im)^\W*{re.escape(s)}\W*:", text)]


def describe_patch(sar_gray, optical_img, purpose="general", terrain=None, season=None, model_id=None,
                   optical_source="translated", max_new_tokens=800, temperature=0.7, top_p=0.8, top_k=20,
                   seed=0):
    """Returns (text, meta). One automatic retry if sections are missing or boilerplate dominates."""
    import torch
    model, proc, used = _load_vlm(model_id)
    facts, fd = sar_stats(sar_gray)
    system, user = build_prompt(purpose, terrain, season, facts, optical_source)
    opt = optical_img.convert("RGB").resize((512, 512), Image.BICUBIC)   # 512 = 16 x 32 px: clean for Qwen-VL patching
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system}]},
        {"role": "user", "content": [
            {"type": "text", "text": "Image 1 - Sentinel-1 SAR:"}, {"type": "image", "image": sar_for_vlm(sar_gray)},
            {"type": "text", "text": "Image 2 - " + ("real Sentinel-2 optical:" if optical_source == "real"
                                                    else "machine-generated optical estimate:")},
            {"type": "image", "image": opt},
            {"type": "text", "text": user}]}]
    inputs = proc.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                      return_dict=True, return_tensors="pt").to(model.device)
    n_in = inputs["input_ids"].shape[1]

    def _gen(sd, temp):
        torch.manual_seed(sd)
        with torch.no_grad():
            ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=temp,
                                 top_p=top_p, top_k=top_k, repetition_penalty=1.05)
        return proc.batch_decode(ids[:, n_in:], skip_special_tokens=True)[0].strip(), int(ids.shape[1] - n_in)

    text, n_new = _gen(seed, temperature)
    retried = False
    if _missing_sections(text, purpose) or _hedge_share(text) > 0.35:
        text, n_new = _gen(seed + 1, min(temperature + 0.15, 0.95)); retried = True
    meta = {"vlm": used, "prompt_version": "v2", "optical_source": optical_source, "retried": retried,
            "truncated": n_new >= max_new_tokens or not re.search(r"[.!?\]]\s*$", text),
            "missing_sections": _missing_sections(text, purpose), "hedge_share": round(_hedge_share(text), 2),
            "sar_facts": facts, "sar_stats": fd}
    return text, meta


def _seed_for(sid):
    return zlib.crc32(sid.encode()) % (2 ** 31)


# ───────────────────────────── entry points ─────────────────────────────
def describe_batch_v2(exp, ids=None, purpose="general", n=None, model_id=None, optical_source="translated",
                      checkpoint_every=10, max_hours=23.0):
    """Resumable, same contract as describe_batch. Output: descriptions/{exp}_{purpose}_v2.json on the Hub.
    optical_source='real' describes the REAL Sentinel-2 image instead (no checkpoint needed; file is
    descriptions/real_{purpose}_v2.json) -> pseudo-reference captions."""
    real = optical_source == "real"
    preflight(); require_hf_token(); sync_data_meta(); seed_all()
    if not real:
        require_checkpoint(exp)
    t0 = time.time(); deadline = t0 + max_hours * 3600
    root, index = load_index()
    ids = ids or load_split("scene")["test"]
    if n: ids = ids[:n]
    tag = "real" if real else exp
    name = f"{tag}_{purpose}_v2"
    out_path = CACHE / "descriptions" / f"{name}.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    hf_pull_file(f"descriptions/{name}.json", out_path, required=False)
    results = json.load(open(out_path)) if out_path.exists() else {}
    todo = [i for i in ids if i not in results]
    log(f"[describe_batch_v2] {name}: {len(results)} done, {len(todo)} to go (of {len(ids)})")
    if not todo:
        return results

    unet = vae = cfg = emb = None
    if not real and exp != "pixel_baseline":
        unet, vae, cfg = load_run(RUNS / exp, dev, tag="best", exp=exp)
    if not real:
        emb = build_text_embeds(dev)
    _load_vlm(model_id)
    try:
        for k, sid in enumerate(todo):
            it = index[sid]
            gray = load_sar_gray(Path(root) / it["s1"])
            if real:
                opt, season, terrain = Image.open(Path(root) / it["s2"]).convert("RGB"), it["season"], it["cls"]
            else:
                opt, season, terrain = translate_sar_to_optical(exp, sample_id=sid, unet=unet, vae=vae, cfg=cfg, emb=emb)
            text, meta = describe_patch(gray, opt, purpose, terrain, season, model_id, optical_source, seed=_seed_for(sid))
            results[sid] = {"id": sid, "exp": tag, "purpose": purpose, "season": season, "terrain": terrain,
                            "description": text, "warning": WARNING, **meta}
            if (k + 1) % checkpoint_every == 0 or k == len(todo) - 1 or time.time() >= deadline:
                json.dump(results, open(out_path, "w"), indent=1)
                hf_push_file(out_path, f"descriptions/{name}.json", quiet=True)
                log(f"  [{k + 1}/{len(todo)}] saved ({sid})")
            if time.time() >= deadline:
                log("[describe_batch_v2] max_hours reached; re-run the same call to resume."); break
    finally:
        del unet, vae, emb
        gc.collect()
        try:
            import torch; torch.cuda.empty_cache()
        except Exception:
            pass
    return results


def describe_uploaded_sar_v2(path, exp="baseline", purpose="general", season=None, terrain=None, model_id=None,
                             out_dir="/content/sar_fyp/descriptions"):
    preflight(); require_checkpoint(exp)
    gray = load_sar_gray(path)
    img, season, terrain = translate_sar_to_optical(exp, sar_image=path, season=season, terrain=terrain)
    text, meta = describe_patch(gray, img, purpose, terrain, season, model_id, seed=_seed_for(str(path)))
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    stem = Path(path).stem
    img.save(out_dir / f"{stem}_optical.png"); sar_for_vlm(gray).save(out_dir / f"{stem}_sar_view.png")
    rec = {"input": str(path), "exp": exp, "purpose": purpose, "season": season, "terrain": terrain,
           "description": text, "warning": WARNING, **meta}
    json.dump(rec, open(out_dir / f"{stem}_description_v2.json", "w"), indent=2)
    print(text); print(f"\nsaved -> {out_dir}/{stem}_*")
    return rec


def show_examples_v2(exp, purpose="general", n=3, seed=0, optical_source="translated"):
    import random, textwrap
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    name = f"{'real' if optical_source == 'real' else exp}_{purpose}_v2"
    path = CACHE / "descriptions" / f"{name}.json"
    hf_pull_file(f"descriptions/{name}.json", path, required=True)
    res = json.load(open(path)); root, index = load_index()
    ids = random.Random(seed).sample(list(res), min(n, len(res)))
    unet = vae = cfg = emb = None
    if optical_source != "real":
        if exp != "pixel_baseline": unet, vae, cfg = load_run(RUNS / exp, dev, tag="best", exp=exp)
        emb = build_text_embeds(dev)
    fig, ax = plt.subplots(len(ids), 3, figsize=(17, 5.2 * len(ids)), gridspec_kw={"width_ratios": [1, 1, 2.3]})
    ax = np.atleast_2d(ax)
    for r, sid in enumerate(ids):
        it = index[sid]; gray = load_sar_gray(Path(root) / it["s1"])
        opt = (Image.open(Path(root) / it["s2"]).convert("RGB") if optical_source == "real" else
               translate_sar_to_optical(exp, sample_id=sid, unet=unet, vae=vae, cfg=cfg, emb=emb)[0])
        ax[r, 0].imshow(sar_for_vlm(gray, 256)); ax[r, 0].set_title(f"{sid}\nSAR (despeckled)", fontsize=8)
        ax[r, 1].imshow(opt); ax[r, 1].set_title("real optical" if optical_source == "real" else "translated optical", fontsize=8)
        ax[r, 2].axis("off")
        ax[r, 2].text(0, 1, "\n".join(textwrap.fill(l, 95) for l in res[sid]["description"].split("\n")),
                      va="top", fontsize=8, family="DejaVu Sans")
        for a in ax[r, :2]: a.set_xticks([]); a.set_yticks([])
    plt.tight_layout()
    out = CACHE / "descriptions" / f"{name}_examples.png"
    plt.savefig(out, dpi=120, bbox_inches="tight"); plt.close()
    try:
        from IPython.display import Image as IPImage, display
        display(IPImage(filename=str(out)))
    except ImportError:
        log(f"saved to {out}")
    return out


# ───────────────────────────── evaluation: are descriptions distinct, specific and faithful? ─────────────────────────────
def _load_named(name):
    p = CACHE / "descriptions" / f"{name}.json"
    hf_pull_file(f"descriptions/{name}.json", p, required=True)
    return json.load(open(p))


def diversity_report(names, ids=None, embed_model="sentence-transformers/all-MiniLM-L6-v2"):
    """names: e.g. ['baseline_general', 'baseline_general_v2'] (file stems under descriptions/ on the Hub).
    Compares on the SAME ids. Lower pairwise similarity = more distinct; lower hedge share = less boilerplate."""
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer(embed_model)
    data = {nm: _load_named(nm) for nm in names}
    common_ids = sorted(set.intersection(*[set(d) for d in data.values()]))
    ids = ids or common_ids
    rows = {}
    for nm, d in data.items():
        txt = [d[i]["description"] for i in ids]
        E = st.encode(txt, normalize_embeddings=True); S = E @ E.T
        iu = np.triu_indices(len(ids), 1)
        words = [t.lower().split() for t in txt]
        bigrams = [b for w in words for b in zip(w, w[1:])]
        rows[nm] = {"n": len(ids), "pairwise_cos": round(float(S[iu].mean()), 3),
                    "distinct_2": round(len(set(bigrams)) / max(1, len(bigrams)), 3),
                    "hedge_sentence_share": round(float(np.mean([_hedge_share(t) for t in txt])), 3),
                    "avg_words": round(float(np.mean([len(w) for w in words])), 1),
                    "truncated_share": round(float(np.mean([not re.search(r"[.!?\]]\s*$", t.strip()) for t in txt])), 3)}
    for nm, r in rows.items(): print(nm, r)
    return rows


def fidelity_report(pred_name, ref_name="real_general_v2", embed_model="sentence-transformers/all-MiniLM-L6-v2"):
    """How close is the description of the TRANSLATED image to the description of the REAL optical image of the
    same patch (made with describe_batch_v2(..., optical_source='real'))? Reports matched-pair similarity vs
    a shuffled-pair baseline; the gap is the image-specific signal."""
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer(embed_model)
    P, R = _load_named(pred_name), _load_named(ref_name)
    ids = sorted(set(P) & set(R))
    a = st.encode([P[i]["description"] for i in ids], normalize_embeddings=True)
    b = st.encode([R[i]["description"] for i in ids], normalize_embeddings=True)
    S = a @ b.T
    matched = float(np.diag(S).mean())
    off = float((S.sum() - np.trace(S)) / (len(ids) * (len(ids) - 1)))
    top1 = float(np.mean(S.argmax(1) == np.arange(len(ids))))
    out = {"n": len(ids), "matched_cos": round(matched, 3), "shuffled_cos": round(off, 3),
           "gap": round(matched - off, 3), "top1_retrieval": round(top1, 3)}
    print(out); return out


In [ ]:
# 2) offline sanity check (no VLM download): prompts + measured SAR facts on one real test patch
from describe_v2 import prompts_are_sane, sar_stats, load_sar_gray, sar_for_vlm
from common import load_index, load_split
from pathlib import Path
assert prompts_are_sane()
root, index = load_index(); sid = load_split("scene")["test"][0]
facts, _ = sar_stats(load_sar_gray(Path(root) / index[sid]["s1"]))
print(sid, index[sid]["cls"], index[sid]["season"]); print(facts)
display(sar_for_vlm(load_sar_gray(Path(root) / index[sid]["s1"]), 384))


In [ ]:
# 3) same 8 ids as your v1 run, new pipeline (resumable, writes descriptions/baseline_general_v2.json)
from describe_v2 import describe_batch_v2, show_examples_v2
describe_batch_v2(EXP, purpose="general", n=8)
show_examples_v2(EXP, purpose="general", n=3)


In [ ]:
# 4) quantify: v1 vs v2 on the same ids (lower pairwise_cos / hedge share = better; truncated_share should be 0)
from describe_v2 import diversity_report
ids8 = __import__("common").load_split("scene")["test"][:8]
diversity_report(["baseline_general", "baseline_general_v2"], ids=ids8)


In [ ]:
# 5) (optional, recommended) pseudo-reference: describe the REAL Sentinel-2 image the same way, then measure
#    whether the description of the TRANSLATED image tracks it better than a random other patch would.
from describe_v2 import describe_batch_v2, fidelity_report
describe_batch_v2(EXP, purpose="general", n=8, optical_source="real")
fidelity_report("baseline_general_v2", "real_general_v2")


In [ ]:
# 6) other purposes once (3) looks good
from describe_v2 import describe_batch_v2
describe_batch_v2(EXP, purpose="flood", n=8)
describe_batch_v2(EXP, purpose="defense", n=8)


In [ ]:
# 7) your own SAR image
from describe_v2 import describe_uploaded_sar_v2
rec = describe_uploaded_sar_v2("/content/your_sar_image.png", exp=EXP, purpose="general", season=None, terrain=None)
